# 04. PyTorch Custom Dataset

This notebook builds an image classification workflow on the Fashion MNIST dataset using PyTorch and TorchVision — covering data loading, a baseline linear model, and a full training/evaluation loop.

- `torchvision.datasets` — provides ready-to-use vision datasets with built-in train/test splits
- `torch.utils.data.DataLoader` — batches and shuffles data for training
- `nn.Sequential` — chains layers into a simple feed-forward model

**Note:** This notebook trains on CPU by default; a CUDA/ROCm device is only used if detected in the setup cell.

**Resources**
1. Notebook: https://www.learnpytorch.io/03_pytorch_computer_vision/
2. Video: https://www.youtube.com/watch?v=V_xro1bcAuA&t=5277s

## Setup & Environment Check

Import the core libraries and detect the available device up front so every tensor and model can be moved to the right place later in the notebook.

- `torch.cuda.is_available()` — checks for a CUDA/ROCm-capable GPU and falls back to CPU otherwise
- `warnings.filterwarnings("ignore")` — suppresses noisy library warnings so the output stays readable
- `DEVICE` — global variable set here and reused throughout the notebook for `.to(DEVICE)` calls

**Note:** This model is trained entirely on `"cpu"` later in the notebook — `DEVICE` is detected here for reference and timing comparisons, not used to move the model or data.

In [1]:
# Capture runtime details so later cells can choose the right device.
import platform
import torch
from torch import nn
import warnings
import numpy as np
import matplotlib.pyplot as plt
import torchvision
from torchvision  import datasets
from torchvision import transforms
from torchvision.transforms import ToTensor

# Keep notebook output focused on the example results.
warnings.filterwarnings("ignore")

print(f"--- System Information ---")
print(f"Platform: {platform.platform()}")
print(f"Python:   {platform.python_version()}")
print(f"PyTorch:  {torch.__version__}")

print(f"\n--- GPU/ROCm Accelerators ---")
cuda_available = torch.cuda.is_available()
print(f"CUDA Available: {cuda_available}")

if cuda_available:
    print(f"Device Count:   {torch.cuda.device_count()}")
    print(f"Primary Device: {torch.cuda.get_device_name(0)}")
    DEVICE = torch.device("cuda:0")
else:
    print("Optimization Note: No CUDA-capable GPU detected. Training and inference will use the CPU.")
    DEVICE = torch.device("cpu")


--- System Information ---
Platform: Windows-11-10.0.26200-SP0
Python:   3.12.13
PyTorch:  2.9.1+rocm7.2.1

--- GPU/ROCm Accelerators ---
CUDA Available: True
Device Count:   1
Primary Device: AMD Radeon RX 7900 XT


In [52]:
# Create a new instance of FashionMNISTModelV2 (the same class as our saved state_dict())
# Note: loading model will error if the shapes here aren't the same as the saved version
loaded_model_2 = FashionMNISTModelV2(input_shape=1, 
                                    hidden_units=10, # try changing this to 128 and seeing what happens 
                                    output_shape=10) 

# Load in the saved state_dict()
loaded_model_2.load_state_dict(torch.load(f=MODEL_SAVE_PATH))

# Send model to GPU
loaded_model_2 = loaded_model_2.to("cpu")

In [53]:
# Evaluate loaded model
torch.manual_seed(42)

loaded_model_2_results = eval_model(
    model=loaded_model_2,
    data_loader=test_dataloader,
    loss_fn=loss_fn, 
    accuracy_fn=accuracy_fn
)

loaded_model_2_results

{'model_name': 'FashionMNISTModelV2',
 'model_loss': 0.3252837061882019,
 'model_acc': 88.07907348242811}

In [54]:
model_2_results

{'model_name': 'FashionMNISTModelV2',
 'model_loss': 0.3252837061882019,
 'model_acc': 88.07907348242811}

## Video Timestamp

Reference point in the source tutorial video for this section of the notebook.

- `~15:52:00` — marks the "Creating DataLoaders" segment, useful for jumping back to the explanation if revisiting this notebook later